# ENERGIZE NILM — TFLite INT8 + Edge TPU Pipeline (TCN)

This notebook converts the **fine-tuned pruned TCN** (output of `03_pruning.ipynb`) to
**TFLite full-integer INT8** and compiles it for the **Google Coral Edge TPU**.

Pipeline:
1. **Rebuild** pruned PyTorch TCN and load fine-tuned weights
2. **Re-implement** the same architecture in TF/Keras (Functional API)
3. **Transfer weights** — map PyTorch `[out, in, kernel]` → TF `[kernel, in, out]`
4. **Validate** — compare TF vs PyTorch predictions (MAE diff should be < 1 mW)
5. **Evaluate TF/Keras float32** — confirm real-world NILM performance on test set before quantizing
6. **Convert** to TFLite full-integer INT8 with representative dataset calibration
7. **Compile** with `edgetpu_compiler` — prints op-coverage report
8. **Evaluate** quantized model on the test set via TFLite CPU interpreter
9. **Export** — append results row to existing Excel, save `.tflite` artifact

> Architecture note: `CausalConv1d` in PyTorch ≡ `Conv1D(padding='causal')` in TF.  
> The gated block (ReLU × Sigmoid) maps directly; skip concatenation is channel-wise.

---
## Google Colab Setup
1. Upload your `ENERGIZE` folder to Google Drive
2. Run the cell below first and edit `DRIVE_PROJECT_PATH`

In [1]:
# ============================================================================
# COLAB SETUP â run this cell first
# ============================================================================
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch_pruning', 'openpyxl', 'tensorflow'])

    # =========================================================================
    DRIVE_PROJECT_PATH = '/content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL'  # <-- EDIT THIS
    # =========================================================================

    import os
    from pathlib import Path
    project_root = Path(DRIVE_PROJECT_PATH)
    if not project_root.exists():
        raise FileNotFoundError(f'Project folder not found: {project_root}')
    os.chdir(project_root)
    sys.path.insert(0, str(project_root))
    print(f'Project root: {project_root}')
else:
    import os
    from pathlib import Path
    project_root = Path(os.getcwd()).parent
    sys.path.insert(0, str(project_root))
    print(f'Running locally. Project root: {project_root}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL


## 1. Imports

In [2]:
import numpy as np
import pandas as pd
import torch

import tensorflow as tf

from src_pytorch import (
    CNN_NILM, TCN_NILM, CNN_NILM_Seq2Seq,
    SimpleNILMDataLoader,
    set_seeds,
    get_model_stats,
    compute_status,
    compute_metrics,
    param_ratio_to_channel_ratio,
    get_appliance_params,
    get_model_config,
    save_pipeline_results,
)
from src_pytorch.quantizer import (
    # TCN
    rebuild_pruned_tcn,
    read_pruned_channels,
    build_tcn_keras,
    transfer_weights,
    # CNN Seq2Point
    rebuild_pruned_cnn,
    read_pruned_cnn_channels,
    build_cnn_keras,
    transfer_cnn_weights,
    # CNN Seq2Seq
    rebuild_pruned_cnn_seq2seq,
    read_pruned_cnn_seq2seq_channels,
    build_cnn_seq2seq_keras,
    transfer_cnn_seq2seq_weights,
    # Shared
    validate_weight_transfer,
    convert_to_tflite_int8,
    compile_edgetpu,
    evaluate_tflite,
    save_predictions_csv,
    upsert_excel_row,
)

set_seeds(42)
pt_device = torch.device('cpu')  # weight export only

print(f'PyTorch    : {torch.__version__}')
print(f'TensorFlow : {tf.__version__}')

Seeds set to 42
PyTorch    : 2.10.0+cu128
TensorFlow : 2.19.0


## 2. Configuration

**Edit only this cell.** `MODEL_NAME`, `APPLIANCE_NAME` and `PRUNING_RATIO` must match
the settings used in `03_pruning.ipynb`.

In [18]:
# ============================================================================
# USER CONFIGURATION  (must match 05_structured_pruning.ipynb settings)
# ============================================================================
MODEL_NAME      = 'cnn_seq2seq'      # 'cnn'  |  'cnn_seq2seq'  |  'tcn'
APPLIANCE_NAME  = 'boiler'   # 'boiler'  |  'ac_1'  |  'washing_machine'
PRUNING_RATIO   = 0.5       # target *parameter* reduction — same value used in 05_structured_pruning.ipynb
FINETUNE_EPOCHS = 1          # must match the value used in 05_structured_pruning.ipynb
N_CALIB_BATCHES = 30        # batches from training set used for INT8 calibration
DATASET_NAME    = 'plegma'

# ============================================================================
# AUTO-DERIVED — do not edit below this line
# ============================================================================
# Model config from src_pytorch (single source of truth — same as notebook 05)
_mcfg = get_model_config(MODEL_NAME)
cfg = {
    'window'          : _mcfg['input_window_length'],
    'batch_size'      : _mcfg['batch_size'],
    'depth'           : _mcfg.get('depth', 9),
    'filters'         : _mcfg.get('nb_filters'),
    'dropout'         : _mcfg.get('dropout', 0.2),
    'stacks'          : _mcfg.get('stacks', 1),
    # Protect the correct output layer: CNN Seq2Point → 1, CNN Seq2Seq / TCN → window
    'args_window_size': 1 if MODEL_NAME == 'cnn' else _mcfg['input_window_length'],
}

# Appliance config from src_pytorch (single source of truth — synced with config.py)
app_cfg = get_appliance_params(DATASET_NAME, APPLIANCE_NAME)

WINDOW                 = cfg['window']
BATCH_SIZE             = cfg['batch_size']
THRESHOLD              = app_cfg['threshold']
CUTOFF                 = app_cfg['cutoff']
MIN_ON                 = app_cfg.get('min_on')
MIN_OFF                = app_cfg.get('min_off')
MIN_COMMITTED_DURATION = app_cfg.get('min_committed_duration')
pct                    = int(PRUNING_RATIO * 100)

if PRUNING_RATIO == 0:
   _channel_ratio = 0
else:
  _channel_ratio = param_ratio_to_channel_ratio(PRUNING_RATIO)

# -- Output directories (shared with 05_structured_pruning.ipynb) -------------
OUT_DIR     = project_root / 'outputs' / f'{MODEL_NAME}_{APPLIANCE_NAME}'
MODELS_DIR  = OUT_DIR / 'models'
PREDS_DIR   = OUT_DIR / 'predictions'
METRICS_DIR = OUT_DIR / 'metrics'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PREDS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH      = OUT_DIR / 'checkpoint' / 'model.pt'
DATA_DIR   = Path('/content/drive/MyDrive/Colab Notebooks/ENERGIZE') / 'data' / 'processed' / DATASET_NAME / APPLIANCE_NAME
EXCEL_PATH     = OUT_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_comparative_results.xlsx'
finetuned_ckpt = (
    MODELS_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_pruned_{pct}pct_finetuned.pt'
    if PRUNING_RATIO > 0 else None
)

print(f'Model                : {MODEL_NAME.upper()}')
print(f'Appliance            : {APPLIANCE_NAME}')
print(f'Target param removal : {pct}%  ->  channel ratio: {_channel_ratio:.4f}')
print(f'Finetune epochs      : {FINETUNE_EPOCHS}')
print(f'Calib batches        : {N_CALIB_BATCHES}')
print(f'Window               : {WINDOW}')
print(f'Threshold            : {THRESHOLD} W  |  Cutoff: {CUTOFF} W')
print(f'Min ON               : {MIN_ON} samples  |  Min OFF: {MIN_OFF} samples')
if MIN_COMMITTED_DURATION:
    print(f'Min committed dur.   : {MIN_COMMITTED_DURATION} samples')
print(f'Baseline ckpt        : {CKPT_PATH}  ({"found" if CKPT_PATH.exists() else "NOT FOUND"})')

print(f'Models dir           : {MODELS_DIR}')
print(f'Predictions          : {PREDS_DIR}')
print(f'Metrics dir          : {METRICS_DIR}')
print(f'Excel                : {EXCEL_PATH}')

Model                : CNN_SEQ2SEQ
Appliance            : boiler
Target param removal : 50%  ->  channel ratio: 0.2929
Finetune epochs      : 1
Calib batches        : 30
Window               : 299
Threshold            : 800 W  |  Cutoff: 4000 W
Min ON               : 30 samples  |  Min OFF: 6 samples
Baseline ckpt        : /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/checkpoint/model.pt  (found)
Models dir           : /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/models
Predictions          : /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/predictions
Metrics dir          : /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/metrics
Excel                : /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/cnn_seq2seq_

## 3. Load Data

In [19]:
data_loader = SimpleNILMDataLoader(
    data_dir=str(DATA_DIR),
    model_name=MODEL_NAME,
    batch_size=BATCH_SIZE,
    input_window_length=WINDOW,
    train=True,
    num_workers=0,
)

print(f'Train batches : {len(data_loader.train)}')
print(f'Val   batches : {len(data_loader.val)}')
print(f'Test  batches : {len(data_loader.test)}')

Train batches : 46335
Val   batches : 9
Test  batches : 11


## 4. Rebuild Pruned PyTorch Model

Reproduces the pruned architecture deterministically for both CNN and TCN:
1. Build a fresh baseline model and load the original checkpoint
2. Re-apply the same pruning (same weights → same channels removed)
3. Load the fine-tuned checkpoint

The resulting `pt_model` is used only to **export weights** to TF/Keras.

In [20]:
print(f'Rebuilding pruned {MODEL_NAME.upper()} model...')
if PRUNING_RATIO == 0:
  print(f'Loading baseline {MODEL_NAME.upper()} (no pruning)...')
  if MODEL_NAME == 'tcn':
      pt_model = TCN_NILM(input_window_length=WINDOW, depth=cfg['depth'],
                            nb_filters=cfg['filters'], dropout=cfg['dropout'], stacks=cfg['stacks'])
  elif MODEL_NAME == 'cnn_seq2seq':
      pt_model = CNN_NILM_Seq2Seq(input_window_length=WINDOW)
  else:
      pt_model = CNN_NILM(input_window_length=WINDOW)
  pt_model.load_state_dict(torch.load(str(CKPT_PATH), map_location=pt_device))
  pt_model.eval()

elif MODEL_NAME == 'tcn':
    pt_model = rebuild_pruned_tcn(
        cfg                 = cfg,
        baseline_ckpt       = CKPT_PATH,
        pruning_ratio       = PRUNING_RATIO,
        finetuned_ckpt_path = finetuned_ckpt,
        device              = pt_device,
    )
elif MODEL_NAME == 'cnn_seq2seq':
    pt_model = rebuild_pruned_cnn_seq2seq(
        cfg                 = cfg,
        baseline_ckpt       = CKPT_PATH,
        pruning_ratio       = PRUNING_RATIO,
        finetuned_ckpt_path = finetuned_ckpt,
        device              = pt_device,
    )
else:  # cnn
    pt_model = rebuild_pruned_cnn(
        cfg                 = cfg,
        baseline_ckpt       = CKPT_PATH,
        pruning_ratio       = PRUNING_RATIO,
        finetuned_ckpt_path = finetuned_ckpt,
        device              = pt_device,
    )

dummy_input = torch.randn(1, WINDOW).to(pt_device)
pruned_params, pruned_macs, pruned_mb = get_model_stats(pt_model, dummy_input)

print(f'\nPruned {MODEL_NAME.upper()} (PyTorch float32):')
print(f'  Parameters : {pruned_params:,}')
print(f'  MACs       : {pruned_macs:,}')
print(f'  Size       : {pruned_mb:.3f} MB')

Rebuilding pruned CNN_SEQ2SEQ model...
Baseline model  — MACs: 24,484,743.0  |  Params: 14,168,899
Pruned model    — MACs: 12,123,437.0  |  Params: 7,077,534
MACs reduction  : 50.5%
Param reduction : 50.0%
Output shape    : torch.Size([1, 299, 1])

Pruned CNN_SEQ2SEQ (PyTorch float32):
  Parameters : 7,077,534
  MACs       : 12,123,437.0
  Size       : 26.999 MB


### 4b. Sanity-check: Evaluate Pruned PyTorch Model

Confirm the fine-tuned pruned PyTorch model achieves expected performance **before** converting
to TF/Keras. This isolates any degradation introduced by the weight transfer.

In [21]:
# from src_pytorch.evaluator import evaluate_model

# print(f'Evaluating pruned PyTorch {MODEL_NAME.upper()} on test set...')
# pt_metrics, pt_gt, pt_pred, pt_gt_status, pt_pred_status = evaluate_model(
#     model                  = pt_model,
#     data_loader            = data_loader,
#     model_name             = MODEL_NAME,
#     cutoff                 = CUTOFF,
#     threshold              = THRESHOLD,
#     device                 = pt_device,
#     input_window_length    = WINDOW,
#     min_on                 = MIN_ON,
#     min_off                = MIN_OFF,
#     min_committed_duration = MIN_COMMITTED_DURATION,
# )

# pt_preds_csv = PREDS_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_pruned_{pct}pct_pytorch_preds.csv'
# save_predictions_csv(pt_gt, pt_pred, pt_preds_csv, gt_status=pt_gt_status, pred_status=pt_pred_status)

# print(f'\nPruned PyTorch {MODEL_NAME.upper()} Results:')
# for k, v in pt_metrics.items():
#     print(f'  {k:<25}: {v:.4f}')

## 5. Build TF/Keras Model

Channel counts are read directly from `pt_model.state_dict()` so the TF model
matches the pruned architecture exactly.

**TCN** — WaveNet-style gated convolutions, Seq2Seq (output = full window):

| PyTorch | TF/Keras |
|---|---|
| `CausalConv1d(k=2, dil=2^i)` | `Conv1D(2, padding='causal', dilation_rate=2^i)` |
| `signal * gate` | `Multiply()` |
| `torch.cat(skips, dim=1)` | `Concatenate(axis=-1)` |

**CNN** — five plain Conv1d + two Dense layers, Seq2Point (output = scalar):

| PyTorch | TF/Keras |
|---|---|
| `Conv1d(k, no padding)` | `Conv1D(k, padding='valid')` |
| `Linear(in, out)` | `Dense(out)` — weights transposed |

TF uses **channel-last** format `(batch, seq_len, channels)` throughout.

In [22]:
if MODEL_NAME == 'tcn':
    initial_ch, block_filters = read_pruned_channels(
        pt_model, depth=cfg['depth'], stacks=cfg['stacks']
    )
    print(f'Pruned initial_conv channels : {initial_ch}  (baseline: {cfg["filters"][0]})')
    print(f'Pruned block channels        : {block_filters}')
    print(f'Baseline block channels      : {cfg["filters"]}')

    tf_model = build_tcn_keras(
        initial_ch    = initial_ch,
        block_filters = block_filters,
        depth         = cfg['depth'],
        stacks        = cfg['stacks'],
        dropout_rate  = cfg['dropout'],
        seq_len       = WINDOW,
    )

elif MODEL_NAME == 'cnn_seq2seq':
    cnn_filters, dense1_units = read_pruned_cnn_seq2seq_channels(pt_model)
    print(f'Pruned CNN Seq2Seq channels : {cnn_filters}  (baseline: [30, 30, 40, 50, 50])')
    print(f'Pruned dense1 units         : {dense1_units}  (baseline: 1024)')

    tf_model = build_cnn_seq2seq_keras(
        filters      = cnn_filters,
        seq_len      = WINDOW,
        dense1_units = dense1_units,
    )

else:  # cnn
    cnn_filters, dense1_units = read_pruned_cnn_channels(pt_model)
    print(f'Pruned CNN channels : {cnn_filters}  (baseline: [30, 30, 40, 50, 50])')
    print(f'Pruned dense1 units : {dense1_units}  (baseline: 1024)')

    tf_model = build_cnn_keras(
        filters      = cnn_filters,
        seq_len      = WINDOW,
        dense1_units = dense1_units,
    )

tf_model.summary(line_length=90)

Pruned CNN Seq2Seq channels : [23, 19, 32, 32, 35]  (baseline: [30, 30, 40, 50, 50])
Pruned dense1 units         : 724  (baseline: 1024)


Model: "CNN_NILM_Seq2Seq_TF"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                          ┃ Output Shape                 ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)                    │ (None, 299, 1)               │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ conv1 (Conv1D)                        │ (None, 290, 23)              │             253 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ relu1 (ReLU)                          │ (None, 290, 23)              │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ conv2 (Conv1D)                        │ (None, 283, 19)              │           3,515 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ relu2 (ReLU)                          │ (None, 283, 19)              │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ conv3 (Conv1D)                        │ (None, 278, 32)              │           3,680 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ relu3 (ReLU)                          │ (None, 278, 32)              │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ conv4 (Conv1D)                        │ (None, 274, 32)              │           5,152 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ relu4 (ReLU)                          │ (None, 274, 32)              │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ drop4 (Dropout)                       │ (None, 274, 32)              │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ conv5 (Conv1D)                        │ (None, 270, 35)              │           5,635 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ relu5 (ReLU)                          │ (None, 270, 35)              │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ drop5 (Dropout)                       │ (None, 270, 35)              │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ permute_flatten (Permute)             │ (None, 35, 270)              │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ flatten (Flatten)                     │ (None, 9450)                 │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ dense1 (Dense)                        │ (None, 724)                  │       6,842,524 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ relu_dense (ReLU)                     │ (None, 724)                  │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ drop_dense (Dropout)                  │ (None, 724)                  │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ dense2 (Dense)                        │ (None, 299)                  │         216,775 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ reshape_output (Reshape)              │ (None, 299, 1)               │               0 │
└───────────────────────────────────────┴──────────────────────────────┴─────────────────┘

 Total params: 7,077,534 (27.00 MB)

 Trainable params: 7,077,534 (27.00 MB)

 Non-trainable params: 0 (0.00 B)

## 6. Transfer PyTorch Weights → TF/Keras

**Conv1d/Conv1D:** `pt[out, in, k]` → `tf[k, in, out]`  (`permute(2, 1, 0)`)  
**Linear/Dense:** `pt[out, in]` → `tf[in, out]`  (transpose)

In [23]:
if MODEL_NAME == 'tcn':
    transfer_weights(tf_model, pt_model.state_dict(), cfg['depth'], cfg['stacks'])
elif MODEL_NAME == 'cnn_seq2seq':
    transfer_cnn_seq2seq_weights(tf_model, pt_model.state_dict())
else:  # cnn
    transfer_cnn_weights(tf_model, pt_model.state_dict())

CNN Seq2Seq weights transferred successfully.


In [24]:
## --- WEIGHT TRANSFER DIAGNOSTIC ---
sd = pt_model.state_dict()

# ── Weight spot-check (model-specific layer names) ────────────────────────────
if MODEL_NAME == 'cnn':
    pt_conv1 = sd['network.0.weight'].permute(2, 1, 0).numpy()
    tf_conv1 = tf_model.get_layer('conv1').get_weights()[0]
    print(f"conv1  match={np.allclose(pt_conv1, tf_conv1, atol=1e-5)}")

    pt_d1 = sd['network.13.weight'].numpy().T
    tf_d1 = tf_model.get_layer('dense1').get_weights()[0]
    print(f"dense1 match={np.allclose(pt_d1, tf_d1, atol=1e-5)}")

elif MODEL_NAME == 'cnn_seq2seq':
    pt_conv1 = sd['network.0.weight'].permute(2, 1, 0).numpy()
    tf_conv1 = tf_model.get_layer('conv1').get_weights()[0]
    print(f"conv1  match={np.allclose(pt_conv1, tf_conv1, atol=1e-5)}")

    pt_d1 = sd['network.13.weight'].numpy().T
    tf_d1 = tf_model.get_layer('dense1').get_weights()[0]
    print(f"dense1 match={np.allclose(pt_d1, tf_d1, atol=1e-5)}")

else:  # tcn
    pt_conv1 = sd['network.initial_conv.weight'].permute(2, 1, 0).numpy()
    tf_conv1 = tf_model.get_layer('initial_conv').get_weights()[0]
    print(f"initial_conv match={np.allclose(pt_conv1, tf_conv1, atol=1e-5)}")

# ── Forward pass comparison (works for Seq2Point and Seq2Seq / TCN) ───────────
sample_pt, _ = next(iter(data_loader.test))
sample_pt = sample_pt[:1]                                            # (1, window)
sample_tf = sample_pt.numpy()[..., np.newaxis].astype(np.float32)   # (1, window, 1)

pt_model.eval()
with torch.no_grad():
    pt_out = pt_model(sample_pt).flatten().numpy()   # (1,) or (window,)

tf_out = tf_model(sample_tf, training=False).numpy().flatten()

diff = np.mean(np.abs(pt_out - tf_out))
status = 'PASS ✓' if diff < 1e-3 else f'FAIL — mean diff={diff:.6f}'
print(f"\nForward pass mean-abs-diff: {diff:.6f}  {status}")

conv1  match=True
dense1 match=True

Forward pass mean-abs-diff: 0.000000  PASS ✓


## 7. Validate â PyTorch vs TF/Keras

Run both models on the same test batch and compare predictions.
A mean absolute difference < 0.001 (normalised) confirms the weight transfer is correct.

In [25]:
print('Validating weight transfer (PyTorch float32 vs TF/Keras float32)...')
val_result = validate_weight_transfer(pt_model, tf_model, data_loader, pt_device)

Validating weight transfer (PyTorch float32 vs TF/Keras float32)...
  Mean |PT − TF|  : 0.00000310  (normalised units)
  Max  |PT − TF|  : 0.00091964  (normalised units)
  Validation      : PASS  (threshold = 0.001)


## 8. Evaluate TF/Keras Float32 Model on Test Set

Run the TF/Keras model (weights already transferred from the fine-tuned pruned PyTorch model)
on the full test split **before** INT8 quantization.

This acts as a **float32 baseline for the TF conversion**, confirming the weight transfer
preserves real-world NILM performance (not just low numerical diff on a single batch).

> No need to re-run the PyTorch evaluation — results from `03_pruning.ipynb` are already in
> the comparative Excel and pipeline CSV.

In [26]:
from tqdm import tqdm

print('Evaluating TF/Keras float32 model on test set...')

all_preds_tf = []
all_gt_tf    = []

for batch_x, batch_y in tqdm(data_loader.test, desc='TF/Keras inference'):
    x_np = batch_x.numpy().astype(np.float32)
    # TF model always expects (batch, window, 1)
    if x_np.ndim == 2:
        x_np = x_np[..., np.newaxis]
    out = tf_model(x_np, training=False).numpy()
    all_preds_tf.append(out.flatten())
    all_gt_tf.append(batch_y.numpy().flatten())

preds_norm_tf = np.concatenate(all_preds_tf)
gt_norm_tf    = np.concatenate(all_gt_tf)

gt_tf   = gt_norm_tf    * CUTOFF
pred_tf = preds_norm_tf * CUTOFF

pred_tf[pred_tf < THRESHOLD] = 0
pred_tf[pred_tf > CUTOFF]    = CUTOFF

keras_metrics = compute_metrics(
    gt_tf, pred_tf, THRESHOLD,
    min_on=MIN_ON, min_off=MIN_OFF, min_committed_duration=MIN_COMMITTED_DURATION,
)

keras_gt_status   = None
keras_pred_status = None
if MIN_ON is not None and MIN_OFF is not None:
    keras_gt_status   = compute_status(gt_tf,   THRESHOLD, MIN_ON, MIN_OFF, MIN_COMMITTED_DURATION)
    keras_pred_status = compute_status(pred_tf, THRESHOLD, MIN_ON, MIN_OFF, MIN_COMMITTED_DURATION)

keras_preds_csv = PREDS_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_pruned_{pct}pct_keras_float32_preds.csv'
save_predictions_csv(
    gt_tf, pred_tf, keras_preds_csv,
    gt_status=keras_gt_status, pred_status=keras_pred_status,
)

print(f'\nTF/Keras Float32 Results:')
for k, v in keras_metrics.items():
    print(f'  {k:<25}: {v:.4f}')

Evaluating TF/Keras float32 model on test set...


TF/Keras inference: 100%|██████████| 11/11 [00:00<00:00, 25.66it/s]


Predictions saved : cnn_seq2seq_boiler_pruned_50pct_keras_float32_preds.csv

TF/Keras Float32 Results:
  mae                      : 8.0886
  accuracy                 : 0.9975
  tp                       : 27538.0000
  tn                       : 1595295.0000
  fp                       : 3678.0000
  fn                       : 348.0000
  total_gt_energy_wh       : 22142.4707
  total_pred_energy_wh     : 22042.2520
  energy_error_percent     : 0.4526
  precision_complex        : 0.8852
  recall_complex           : 0.9851
  f1_complex               : 0.9325


## 9. Convert to TFLite Full-Integer INT8

Three-step process:
1. **Representative dataset** — `N_CALIB_BATCHES` training samples for activation range calibration
2. **Convert** — `TFLITE_BUILTINS_INT8` quantises both weights and activations
3. **Save** `.tflite` file

In [27]:
tflite_path = MODELS_DIR / (
    f'{MODEL_NAME}_{APPLIANCE_NAME}_baseline_int8.tflite'
    if PRUNING_RATIO == 0
    else f'{MODEL_NAME}_{APPLIANCE_NAME}_pruned_{pct}pct_int8.tflite'
)

tflite_path = convert_to_tflite_int8(
    tf_model        = tf_model,
    data_loader     = data_loader,
    window          = WINDOW,
    n_calib_batches = N_CALIB_BATCHES,
    out_path        = tflite_path,
)

tflite_mb = round(tflite_path.stat().st_size / (1024 ** 2), 3)
print(f'Size reduction : {(1 - tflite_mb / pruned_mb) * 100:.1f}%  '
      f'({pruned_mb:.3f} MB â {tflite_mb:.3f} MB)')

Converting to TFLite INT8 (calibrating with 30 batches)...
Saved artifact at '/tmp/tmpndvyglun'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 299, 1), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(None, 299, 1), dtype=tf.float32, name=None)
Captures:
  139459200283088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139459200283472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139459200284048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139459200282704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139459200284240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139459200284816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139459200281744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139459200283664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139459200283280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139459200278288: TensorSpec(shap

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
Calibrating: 100%|██████████| 30/30 [00:56<00:00,  1.89s/batch]


Full-integer INT8 conversion succeeded.

TFLite INT8 model saved : /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/models/cnn_seq2seq_boiler_pruned_50pct_int8.tflite
File size               : 6.790 MB
Size reduction : 74.9%  (26.999 MB â 6.790 MB)


## 10. Compile for Edge TPU

The `edgetpu_compiler` analyses the `.tflite` graph and reports:
- **Ops mapped to the TPU** — INT8 ops with supported kernels run at ~4 TOPS
- **Ops falling back to CPU** — unsupported ops (e.g. sigmoid, element-wise ×)

Even with CPU fallback ops, the compiled model is valid for evaluation.

> Install: `sudo apt install edgetpu-compiler` (Linux/Colab) or download from  
> [coral.ai/docs/edgetpu/compiler](https://coral.ai/docs/edgetpu/compiler/)

In [29]:
edgetpu_path = compile_edgetpu(tflite_path, out_dir=MODELS_DIR)

edgetpu_compiler not found in PATH.
Install from https://coral.ai/docs/edgetpu/compiler/

Skipping compilation — the TFLite INT8 model can still be evaluated on CPU.


## 11. Evaluate TFLite INT8 Model (CPU Interpreter)

The TFLite CPU interpreter runs the quantized model sample-by-sample:
1. **Quantize input** float32 → int8 using the input tensor's scale / zero-point
2. **Invoke** the interpreter
3. **Dequantize output** int8 → float32
4. Accumulate predictions and compute NILM metrics

In [30]:
print('Evaluating TFLite INT8 model on CPU interpreter...')
tflite_metrics, gt, pred, gt_status, pred_status = evaluate_tflite(
    tflite_path            = tflite_path,
    data_loader            = data_loader,
    window                 = WINDOW,
    cutoff                 = CUTOFF,
    threshold              = THRESHOLD,
    min_on                 = MIN_ON,
    min_off                = MIN_OFF,
    min_committed_duration = MIN_COMMITTED_DURATION,
)

preds_csv = PREDS_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_pruned_{pct}pct_int8_preds.csv'
save_predictions_csv(gt, pred, preds_csv, gt_status=gt_status, pred_status=pred_status)

print(f'\nTFLite INT8 Results:')
for k, v in tflite_metrics.items():
    print(f'  {k:<25}: {v:.4f}')

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Evaluating TFLite INT8 model on CPU interpreter...
  Input  dtype / shape : int8 [  1 299   1]
  Output dtype / shape : int8 [  1 299   1]
  Input  quant params  : scale=0.050810, zero_point=-119
  Output quant params  : scale=0.004005, zero_point=-97


TFLite inference: 100%|██████████| 11/11 [00:02<00:00,  4.30it/s]


Predictions saved : cnn_seq2seq_boiler_pruned_50pct_int8_preds.csv

TFLite INT8 Results:
  mae                      : 8.3133
  accuracy                 : 0.9974
  tp                       : 27555.0000
  tn                       : 1595050.0000
  fp                       : 3923.0000
  fn                       : 331.0000
  total_gt_energy_wh       : 22142.4707
  total_pred_energy_wh     : 22135.5332
  energy_error_percent     : 0.0313
  precision_complex        : 0.8776
  recall_complex           : 0.9861
  f1_complex               : 0.9287


## 12. Save Results to Excel

In [31]:
EXCEL_PATH

PosixPath('/content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/cnn_seq2seq_boiler_comparative_results.xlsx')

In [32]:
new_row = {
    'Model'           : f'{MODEL_NAME.upper()} Baseline + TFLite INT8' if PRUNING_RATIO == 0
                        else f'{MODEL_NAME.upper()} Pruned {pct}% + FT + TFLite INT8',
    'Architecture'      : MODEL_NAME.upper(),
    'Appliance'         : APPLIANCE_NAME,
    'Pruning_Ratio_%' : 0 if PRUNING_RATIO == 0 else pct,
    'Params'            : pruned_params,
    'MACs'              : pruned_macs,
    'MB'                : tflite_mb,
    'MAE'               : round(tflite_metrics['mae'],                              4),
    'F1_Complex'        : round(tflite_metrics['f1_complex'],          4) if 'f1_complex'        in tflite_metrics else '',
    'Precision_Complex' : round(tflite_metrics['precision_complex'],   4) if 'precision_complex' in tflite_metrics else '',
    'Recall_Complex'    : round(tflite_metrics['recall_complex'],      4) if 'recall_complex'    in tflite_metrics else '',
    'Accuracy'          : round(tflite_metrics['accuracy'],                         4),
    'GT_Energy_Wh'      : round(tflite_metrics['total_gt_energy_wh'],              2),
    'Pred_Energy_Wh'    : round(tflite_metrics['total_pred_energy_wh'],            2),
}

upsert_excel_row(new_row, EXCEL_PATH)

Upserting row in existing Excel: /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/cnn_seq2seq_boiler_comparative_results.xlsx
Excel updated: /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/cnn_seq2seq_boiler_comparative_results.xlsx


In [22]:
# Upsert TFLite metrics into the pipeline metrics CSV (appends / overwrites this stage)
pipeline_csv = METRICS_DIR / f'{APPLIANCE_NAME}_{MODEL_NAME}_pipeline_results.csv'

tflite_stage_label = f'Pruned {pct}% + FT + TFLite INT8'
new_row_df = pd.DataFrame([{
    'Stage'             : tflite_stage_label,
    'MAE'               : round(tflite_metrics['mae'],                              4),
    'F1_Complex'        : round(tflite_metrics['f1_complex'],          4) if 'f1_complex'        in tflite_metrics else '',
    'Precision_Complex' : round(tflite_metrics['precision_complex'],   4) if 'precision_complex' in tflite_metrics else '',
    'Recall_Complex'    : round(tflite_metrics['recall_complex'],      4) if 'recall_complex'    in tflite_metrics else '',
    'Accuracy'          : round(tflite_metrics['accuracy'],                         4),
    'GT_Energy_Wh'      : round(tflite_metrics['total_gt_energy_wh'],              2),
    'Pred_Energy_Wh'    : round(tflite_metrics['total_pred_energy_wh'],            2),
}])

if pipeline_csv.exists():
    existing = pd.read_csv(pipeline_csv)
    existing = existing[existing['Stage'] != tflite_stage_label]  # remove stale row
    updated  = pd.concat([existing, new_row_df], ignore_index=True)
else:
    updated = new_row_df

updated.to_csv(pipeline_csv, index=False)
print(f'Pipeline metrics updated: {pipeline_csv}')
updated

Pipeline metrics updated: /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_boiler/metrics/boiler_cnn_pipeline_results.csv


,Stage,MAE,F1,F1_Complex,Precision,Recall,Accuracy,Energy_Error_%,Precision_Complex,Recall_Complex,GT_Energy_Wh,Pred_Energy_Wh
0,CNN Baseline,7.3070,0.9298,0.9387,0.8791,0.9868,0.9974,1.39,NaN,NaN,NaN,NaN
1,CNN Pruned 75%,48.9987,0.0000,0.0000,0.0000,0.0000,0.9829,100.00,NaN,NaN,NaN,NaN
2,CNN Pruned 75% + Fine-tuned 1ep,10.3736,0.8936,0.8965,0.8189,0.9834,0.9960,10.26,NaN,NaN,NaN,NaN
3,Pruned 75% + FT + TFLite INT8,8.5638,0.9135,0.9178,0.8483,0.9895,0.9968,8.15,NaN,NaN,NaN,NaN
4,Pruned 0% + FT + TFLite INT8,6.8610,NaN,0.9447,NaN,NaN,0.9976,NaN,0.8998,0.9944,22142.470703,22580.160156


## 13. Summary

In [24]:
C, W = 30, 18
SEP  = '=' * (C + W * 2 + 2)
sep  = '-' * (C + W * 2 + 2)
hfmt = f'{{:<{C}}} {{:>{W}}} {{:>{W}}}'
rfmt = f'{{:<{C}}} {{:>{W}}} {{:>{W}}}'

col_ft     = f'Pruned {pct}% + FT (PT)'
col_tflite = 'TFLite INT8'

print(SEP)
print(f'TFLITE SUMMARY  |  {MODEL_NAME.upper()}  |  {APPLIANCE_NAME}  |  pruning={pct}%')
print(SEP)
print(hfmt.format('Metric', col_ft, col_tflite))
print(sep)

# Recover fine-tuned metrics from the shared Excel
ft_label = f'{MODEL_NAME.upper()} Pruned {pct}% + Fine-tuned {FINETUNE_EPOCHS}ep'
ft_mb = ft_mae = ft_f1c = ft_prec_c = ft_rec_c = ft_acc = float('nan')
ft_gt_energy = ft_pred_energy = float('nan')

if EXCEL_PATH.exists():
    _df = pd.read_excel(EXCEL_PATH)
    _ft = _df[_df['Model'] == ft_label]
    if not _ft.empty:
        _ft           = _ft.iloc[0]
        ft_mb         = _ft.get('MB',                float('nan'))
        ft_mae        = _ft.get('MAE',               float('nan'))
        ft_f1c        = _ft.get('F1_Complex',        float('nan'))
        ft_prec_c     = _ft.get('Precision_Complex', float('nan'))
        ft_rec_c      = _ft.get('Recall_Complex',    float('nan'))
        ft_acc        = _ft.get('Accuracy',          float('nan'))
        ft_gt_energy  = _ft.get('GT_Energy_Wh',      float('nan'))
        ft_pred_energy= _ft.get('Pred_Energy_Wh',    float('nan'))


def rrow(label, a, b):
    print(rfmt.format(label, str(a), str(b)))


rrow('MB',                f'{ft_mb:.3f}',    f'{tflite_mb:.3f}')
print(sep)
rrow('MAE (W)',           f'{ft_mae:.4f}',   f'{tflite_metrics["mae"]:.4f}')
rrow('F1 Complex',        f'{ft_f1c:.4f}',   f'{tflite_metrics.get("f1_complex",        float("nan")):.4f}')
rrow('Precision Complex', f'{ft_prec_c:.4f}',f'{tflite_metrics.get("precision_complex", float("nan")):.4f}')
rrow('Recall Complex',    f'{ft_rec_c:.4f}', f'{tflite_metrics.get("recall_complex",    float("nan")):.4f}')
rrow('Accuracy',          f'{ft_acc:.4f}',   f'{tflite_metrics["accuracy"]:.4f}')
rrow('GT Energy Wh',      f'{ft_gt_energy:.2f}',   f'{tflite_metrics.get("total_gt_energy_wh",   float("nan")):.2f}')
rrow('Pred Energy Wh',    f'{ft_pred_energy:.2f}',  f'{tflite_metrics.get("total_pred_energy_wh", float("nan")):.2f}')
print(SEP)
print(f'TFLite INT8 : {tflite_path.name}')
if 'edgetpu_path' in dir() and edgetpu_path is not None and edgetpu_path.exists():
    print(f'Edge TPU    : {edgetpu_path.name}')
print(f'Excel       : {EXCEL_PATH}')
print(f'Preds CSV   : {preds_csv.name}')
print(SEP)

TFLITE SUMMARY  |  CNN  |  boiler  |  pruning=0%
Metric                         Pruned 0% + FT (PT)        TFLite INT8
--------------------------------------------------------------------
MB                                            nan             13.261
--------------------------------------------------------------------
MAE (W)                                       nan             6.8610
F1 Complex                                    nan             0.9447
Precision Complex                             nan             0.8998
Recall Complex                                nan             0.9944
Accuracy                                      nan             0.9976
GT Energy Wh                                  nan           22142.47
Pred Energy Wh                                nan           22580.15
TFLite INT8 : cnn_boiler_baseline_int8.tflite
Excel       : /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_boiler/cnn_boiler_comparative_results.xlsx
